# AWS DE Deep — Glue Catalog, Glue ETL, EMR Serverless, Athena

## Mental model

This notebook walks a realistic AWS data engineering path for a Citi-style telemetry platform:

- **Glue Data Catalog** = central metadata layer for S3 analytics datasets
- **Glue ETL** = managed serverless ETL definition and script packaging
- **EMR Serverless** = Spark without cluster management
- **Athena** = pay-per-scan SQL over S3, where **partitioning is the main cost lever**

### Fixed context used in this notebook

**AWS**
- profile: `study`
- region: `us-east-1`
- account: `357811130281`

**Local PostgreSQL source**
- host: `localhost`
- port: `5432`
- db: `de_telemetry`
- user: `de_admin`
- password: `DeAdmin2026!`

**Tables**
- `endpoints`: 10,000 rows
- `metrics`: 500,000 rows
- `alerts`: 25,000 rows

**Narrative**
- 6,000+ API endpoints monitored for latency, error rate, throughput
- alerts escalate through severity tiers


In [ ]:
import os
import io
import time
import json
import uuid
from datetime import datetime, timezone

import boto3
import pandas as pd
import psycopg2
from botocore.exceptions import ClientError

# No pip installs. Packages are assumed to exist in the target environment.
os.environ["AWS_PROFILE"] = "study"
AWS_PROFILE = os.environ["AWS_PROFILE"]
AWS_REGION = "us-east-1"
AWS_ACCOUNT_ID = "357811130281"

session = boto3.Session(profile_name=AWS_PROFILE, region_name=AWS_REGION)
sts = session.client("sts")
identity = sts.get_caller_identity()

print("AWS profile:", AWS_PROFILE)
print("AWS region :", AWS_REGION)
print("AWS acct   :", identity["Account"])
assert identity["Account"] == AWS_ACCOUNT_ID, f"Expected account {AWS_ACCOUNT_ID}, got {identity['Account']}"


In [ ]:
PG_CONFIG = {
    "host": "localhost",
    "port": 5432,
    "dbname": "de_telemetry",
    "user": "de_admin",
    "password": "DeAdmin2026!",
}

def get_pg_connection():
    return psycopg2.connect(**PG_CONFIG)

def read_sql_df(sql: str) -> pd.DataFrame:
    with get_pg_connection() as conn:
        return pd.read_sql(sql, conn)

def qname(name: str) -> str:
    return name.replace("-", "_").replace("/", "_")

RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
SUFFIX = uuid.uuid4().hex[:8]

S3_BUCKET = f"citi-deep-{AWS_ACCOUNT_ID}-{AWS_REGION}-{SUFFIX}"
S3_PREFIX = f"aws-deep/{RUN_ID}"
RAW_PREFIX = f"{S3_PREFIX}/raw"
ATHENA_PREFIX = f"{S3_PREFIX}/athena"
RESULTS_PREFIX = f"{S3_PREFIX}/results"
GLUE_SCRIPT_PREFIX = f"{S3_PREFIX}/glue-scripts"
PARQUET_PREFIX = f"{S3_PREFIX}/parquet"

GLUE_DB = "citi_deep_glue"
GLUE_ENDPOINTS_TABLE = "endpoints_csv"
ATHENA_DB = "citi_deep_athena"
ATHENA_ALERTS_TABLE = "alerts_partitioned"
ATHENA_ENDPOINTS_TABLE = "endpoints_csv"
WORKGROUP = "citi-analytics"

S3_URI = f"s3://{S3_BUCKET}"
RAW_S3_URI = f"{S3_URI}/{RAW_PREFIX}"
ATHENA_RESULTS_URI = f"{S3_URI}/{RESULTS_PREFIX}/"

print("Run id        :", RUN_ID)
print("Bucket        :", S3_BUCKET)
print("Root S3 URI   :", S3_URI)
print("Glue DB       :", GLUE_DB)
print("Athena DB     :", ATHENA_DB)
print("Workgroup     :", WORKGROUP)


## 1) Setup AWS clients and create an isolated S3 bucket

Everything below is idempotent where possible, and resource names are scoped to this notebook run.


In [ ]:
s3 = session.client("s3")
glue = session.client("glue")
athena = session.client("athena")
emr_serverless = session.client("emr-serverless")

def create_bucket(bucket_name: str, region: str):
    if region == "us-east-1":
        s3.create_bucket(Bucket=bucket_name)
    else:
        s3.create_bucket(
            Bucket=bucket_name,
            CreateBucketConfiguration={"LocationConstraint": region},
        )

create_bucket(S3_BUCKET, AWS_REGION)

for prefix in [RAW_PREFIX, ATHENA_PREFIX, RESULTS_PREFIX, GLUE_SCRIPT_PREFIX, PARQUET_PREFIX]:
    s3.put_object(Bucket=S3_BUCKET, Key=f"{prefix}/")

print("Created bucket and prefixes successfully.")


## 2) Pull source data from PostgreSQL

We keep the dataset context exactly as requested and export the relevant source data for S3/Athena/Glue.


In [ ]:
endpoints_df = read_sql_df(
    '''
    SELECT endpoint_id, name, region, status, category
    FROM endpoints
    ORDER BY endpoint_id
    '''
)

alerts_df = read_sql_df(
    '''
    SELECT alert_id, endpoint_id, severity, message, created_at
    FROM alerts
    ORDER BY alert_id
    '''
)

print("endpoints rows:", len(endpoints_df))
print("alerts rows   :", len(alerts_df))

assert len(endpoints_df) == 10000, f"Expected 10000 endpoints rows, got {len(endpoints_df)}"
assert len(alerts_df) == 25000, f"Expected 25000 alerts rows, got {len(alerts_df)}"

display(endpoints_df.head())
display(alerts_df.head())


In [ ]:
def put_df_as_csv(df: pd.DataFrame, bucket: str, key: str):
    csv_bytes = df.to_csv(index=False).encode("utf-8")
    s3.put_object(Bucket=bucket, Key=key, Body=csv_bytes, ContentType="text/csv")
    return f"s3://{bucket}/{key}"

endpoints_key = f"{RAW_PREFIX}/endpoints/endpoints.csv"
alerts_base_prefix = f"{ATHENA_PREFIX}/alerts_partitioned"
endpoints_s3_uri = put_df_as_csv(endpoints_df, S3_BUCKET, endpoints_key)

# Write partitioned alert CSV files by severity for Athena
partition_keys = []
for severity, part_df in alerts_df.groupby("severity"):
    sev = str(severity).strip().upper()
    key = f"{alerts_base_prefix}/severity={sev}/alerts_{sev}.csv"
    put_df_as_csv(part_df, S3_BUCKET, key)
    partition_keys.append((sev, key, len(part_df)))

print("Endpoints CSV :", endpoints_s3_uri)
print("Alert partitions written:")
for sev, key, row_count in sorted(partition_keys):
    print(f"  severity={sev:<10} rows={row_count:<6} s3://{S3_BUCKET}/{key}")


## 3) Glue Catalog

Create the Glue database `citi_deep_glue`, register a CSV table for `endpoints`, and print the catalog hierarchy.


In [ ]:
def ensure_glue_database(database_name: str):
    try:
        glue.create_database(
            DatabaseInput={
                "Name": database_name,
                "Description": "Citi DE deep dive catalog for endpoints/alerts demo",
            }
        )
        print(f"Created Glue database: {database_name}")
    except ClientError as e:
        if e.response["Error"]["Code"] == "AlreadyExistsException":
            print(f"Glue database already exists: {database_name}")
        else:
            raise

ensure_glue_database(GLUE_DB)

endpoints_table_input = {
    "Name": GLUE_ENDPOINTS_TABLE,
    "Description": "Endpoints CSV registered in Glue Catalog",
    "TableType": "EXTERNAL_TABLE",
    "Parameters": {
        "classification": "csv",
        "delimiter": ",",
        "typeOfData": "file",
        "skip.header.line.count": "1",
    },
    "StorageDescriptor": {
        "Columns": [
            {"Name": "endpoint_id", "Type": "int"},
            {"Name": "name", "Type": "string"},
            {"Name": "region", "Type": "string"},
            {"Name": "status", "Type": "string"},
            {"Name": "category", "Type": "string"},
        ],
        "Location": f"s3://{S3_BUCKET}/{RAW_PREFIX}/endpoints/",
        "InputFormat": "org.apache.hadoop.mapred.TextInputFormat",
        "OutputFormat": "org.apache.hadoop.hive.ql.io.HiveIgnoreKeyTextOutputFormat",
        "Compressed": False,
        "NumberOfBuckets": -1,
        "SerdeInfo": {
            "SerializationLibrary": "org.apache.hadoop.hive.serde2.lazy.LazySimpleSerDe",
            "Parameters": {
                "field.delim": ",",
                "skip.header.line.count": "1",
            },
        },
        "StoredAsSubDirectories": False,
    },
}

try:
    glue.create_table(DatabaseName=GLUE_DB, TableInput=endpoints_table_input)
    print(f"Created Glue table: {GLUE_DB}.{GLUE_ENDPOINTS_TABLE}")
except ClientError as e:
    if e.response["Error"]["Code"] == "AlreadyExistsException":
        print(f"Glue table already exists: {GLUE_DB}.{GLUE_ENDPOINTS_TABLE}")
    else:
        raise

dbs = glue.get_databases()["DatabaseList"]
tables = glue.get_tables(DatabaseName=GLUE_DB)["TableList"]

print("\nGlue Catalog hierarchy")
print(f"- database: {GLUE_DB}")
for t in tables:
    print(f"  - table: {t['Name']}")
    print(f"    - location: {t['StorageDescriptor']['Location']}")
    print(f"    - columns : {[c['Name'] + ':' + c['Type'] for c in t['StorageDescriptor']['Columns']]}")


## 4) Glue ETL Job Definition

We will **define** a Glue ETL job but **not run it** to avoid unnecessary spend.

The ETL script:
- reads CSV alerts from S3
- filters `HIGH` and `CRITICAL`
- writes Parquet output

### DPU pricing mental model
Glue job cost is driven by job runtime and worker/DPU allocation. In practice, the key levers are:
- prune input data early
- use partitioned layouts
- right-size workers
- avoid long idle runtime

This notebook creates the job definition and uploads the ETL script to S3, but intentionally does not start the job.


In [ ]:
GLUE_JOB_NAME = qname(f"citi_alerts_filter_job_{SUFFIX}")
GLUE_SCRIPT_KEY = f"{GLUE_SCRIPT_PREFIX}/citi_alerts_filter_job.py"
GLUE_SCRIPT_S3_URI = f"s3://{S3_BUCKET}/{GLUE_SCRIPT_KEY}"

glue_job_script = f'''
import sys
from awsglue.utils import getResolvedOptions
from pyspark.context import SparkContext
from pyspark.sql import functions as F
from awsglue.context import GlueContext
from awsglue.job import Job

args = getResolvedOptions(sys.argv, ['JOB_NAME', 'INPUT_PATH', 'OUTPUT_PATH'])
sc = SparkContext()
glueContext = GlueContext(sc)
spark = glueContext.spark_session
job = Job(glueContext)
job.init(args['JOB_NAME'], args)

df = (
    spark.read
    .option("header", "true")
    .csv(args['INPUT_PATH'])
)

filtered = (
    df.filter(F.upper(F.col("severity")).isin("HIGH", "CRITICAL"))
)

(
    filtered.write
    .mode("overwrite")
    .parquet(args['OUTPUT_PATH'])
)

job.commit()
'''.strip()

s3.put_object(
    Bucket=S3_BUCKET,
    Key=GLUE_SCRIPT_KEY,
    Body=glue_job_script.encode("utf-8"),
    ContentType="text/x-python",
)

iam = session.client("iam")
role_name = "AWSGlueServiceRole"
try:
    glue_role_arn = iam.get_role(RoleName=role_name)["Role"]["Arn"]
except ClientError:
    # Fallback to the common service-role path if list/read is restricted.
    glue_role_arn = f"arn:aws:iam::{AWS_ACCOUNT_ID}:role/service-role/{role_name}"

default_arguments = {
    "--job-language": "python",
    "--TempDir": f"{S3_URI}/{S3_PREFIX}/temp/",
    "--enable-metrics": "true",
    "--INPUT_PATH": f"s3://{S3_BUCKET}/{ATHENA_PREFIX}/alerts_partitioned/",
    "--OUTPUT_PATH": f"s3://{S3_BUCKET}/{PARQUET_PREFIX}/high_critical_alerts/",
}

job_payload = {
    "Name": GLUE_JOB_NAME,
    "Role": glue_role_arn,
    "ExecutionProperty": {"MaxConcurrentRuns": 1},
    "Command": {
        "Name": "glueetl",
        "ScriptLocation": GLUE_SCRIPT_S3_URI,
        "PythonVersion": "3",
    },
    "DefaultArguments": default_arguments,
    "GlueVersion": "4.0",
    "NumberOfWorkers": 2,
    "WorkerType": "G.1X",
    "Description": "Filter HIGH/CRITICAL alerts and write Parquet",
    "Timeout": 10,
    "MaxRetries": 0,
}

try:
    glue.create_job(**job_payload)
    print("Created Glue job:", GLUE_JOB_NAME)
except ClientError as e:
    if e.response["Error"]["Code"] == "IdempotentParameterMismatchException":
        raise
    if e.response["Error"]["Code"] == "AlreadyExistsException":
        print("Glue job already exists:", GLUE_JOB_NAME)
    else:
        raise

job_def = glue.get_job(JobName=GLUE_JOB_NAME)["Job"]
print("Glue job definition:")
print(json.dumps({
    "Name": job_def["Name"],
    "Role": job_def["Role"],
    "GlueVersion": job_def.get("GlueVersion"),
    "WorkerType": job_def.get("WorkerType"),
    "NumberOfWorkers": job_def.get("NumberOfWorkers"),
    "Command": job_def["Command"],
    "DefaultArguments": job_def.get("DefaultArguments", {}),
}, indent=2))


## 5) EMR Serverless

This section demonstrates the control plane shape without creating an application or submitting a job. That keeps the notebook cost-safe while still grounding the mental model.

**Mental model**
- Glue ETL is opinionated managed ETL
- EMR Serverless is more flexible Spark/Hive execution without cluster management
- Use EMR Serverless when you want Spark power without persistent clusters


In [ ]:
release_labels = emr_serverless.list_release_labels(maxResults=10)["releaseLabels"]
print("Available EMR Serverless release labels:")
for label in release_labels[:10]:
    print(" -", label)

print("\nNo EMR Serverless application is created in this notebook to avoid unnecessary cost.")


## 6) Athena Advanced

We will:
1. create an Athena database
2. create an external CSV table for `endpoints`
3. create a **partitioned** table for `alerts` by `severity`
4. run `MSCK REPAIR TABLE`
5. run 3 partition-aware queries
6. compare scan cost behavior from query stats


In [ ]:
def athena_wait(query_execution_id: str, sleep_seconds: float = 1.5, max_attempts: int = 120):
    for _ in range(max_attempts):
        resp = athena.get_query_execution(QueryExecutionId=query_execution_id)
        status = resp["QueryExecution"]["Status"]["State"]
        if status in {"SUCCEEDED", "FAILED", "CANCELLED"}:
            return resp
        time.sleep(sleep_seconds)
    raise TimeoutError(f"Athena query did not finish in time: {query_execution_id}")

def run_athena(sql: str, database: str = None, workgroup: str = None):
    params = {
        "QueryString": sql,
        "ResultConfiguration": {"OutputLocation": ATHENA_RESULTS_URI},
    }
    if database:
        params["QueryExecutionContext"] = {"Database": database}
    if workgroup:
        params["WorkGroup"] = workgroup

    qid = athena.start_query_execution(**params)["QueryExecutionId"]
    final_resp = athena_wait(qid)
    qexec = final_resp["QueryExecution"]
    state = qexec["Status"]["State"]
    if state != "SUCCEEDED":
        raise RuntimeError(f"Athena query failed: {state} | {qexec['Status'].get('StateChangeReason', '')}")
    return qid, qexec

def bytes_to_mb(n: int) -> float:
    return round(n / (1024 * 1024), 4)

create_db_sql = f"CREATE DATABASE IF NOT EXISTS {ATHENA_DB}"
run_athena(create_db_sql)

create_endpoints_sql = f'''
CREATE EXTERNAL TABLE IF NOT EXISTS {ATHENA_DB}.{ATHENA_ENDPOINTS_TABLE} (
  endpoint_id INT,
  name STRING,
  region STRING,
  status STRING,
  category STRING
)
ROW FORMAT SERDE 'org.apache.hadoop.hive.serde2.OpenCSVSerde'
WITH SERDEPROPERTIES (
  'separatorChar' = ',',
  'quoteChar' = '"',
  'escapeChar' = '\\'
)
STORED AS TEXTFILE
LOCATION 's3://{S3_BUCKET}/{RAW_PREFIX}/endpoints/'
TBLPROPERTIES ('skip.header.line.count'='1')
'''.strip()

run_athena(create_endpoints_sql, database=ATHENA_DB)

create_alerts_partitioned_sql = f'''
CREATE EXTERNAL TABLE IF NOT EXISTS {ATHENA_DB}.{ATHENA_ALERTS_TABLE} (
  alert_id INT,
  endpoint_id INT,
  message STRING,
  created_at TIMESTAMP
)
PARTITIONED BY (severity STRING)
ROW FORMAT SERDE 'org.apache.hadoop.hive.serde2.OpenCSVSerde'
WITH SERDEPROPERTIES (
  'separatorChar' = ',',
  'quoteChar' = '"',
  'escapeChar' = '\\'
)
STORED AS TEXTFILE
LOCATION 's3://{S3_BUCKET}/{ATHENA_PREFIX}/alerts_partitioned/'
TBLPROPERTIES ('skip.header.line.count'='1')
'''.strip()

run_athena(create_alerts_partitioned_sql, database=ATHENA_DB)

repair_qid, repair_exec = run_athena(
    f"MSCK REPAIR TABLE {ATHENA_ALERTS_TABLE}",
    database=ATHENA_DB
)

print("Athena database/table setup complete.")
print("MSCK REPAIR TABLE qid:", repair_qid)


In [ ]:
queries = {
    "q1_high_count": f'''
        SELECT severity, COUNT(*) AS alert_count
        FROM {ATHENA_ALERTS_TABLE}
        WHERE severity = 'HIGH'
        GROUP BY severity
    ''',
    "q2_critical_recent": f'''
        SELECT severity, COUNT(*) AS critical_alerts
        FROM {ATHENA_ALERTS_TABLE}
        WHERE severity = 'CRITICAL'
        GROUP BY severity
    ''',
    "q3_join_hot_regions": f'''
        SELECT e.region, a.severity, COUNT(*) AS alert_count
        FROM {ATHENA_ALERTS_TABLE} a
        JOIN {ATHENA_ENDPOINTS_TABLE} e
          ON a.endpoint_id = e.endpoint_id
        WHERE a.severity IN ('HIGH', 'CRITICAL')
        GROUP BY e.region, a.severity
        ORDER BY alert_count DESC
        LIMIT 20
    ''',
    "q4_unpartitioned_style_scan": f'''
        SELECT COUNT(*) AS alert_count
        FROM {ATHENA_ALERTS_TABLE}
    ''',
}

query_stats = []
for name, sql in queries.items():
    qid, qexec = run_athena(sql, database=ATHENA_DB)
    stats = qexec.get("Statistics", {})
    scanned = stats.get("DataScannedInBytes", 0)
    query_stats.append({
        "query_name": name,
        "query_execution_id": qid,
        "bytes_scanned": scanned,
        "mb_scanned": bytes_to_mb(scanned),
        "engine_ms": stats.get("EngineExecutionTimeInMillis"),
        "total_ms": stats.get("TotalExecutionTimeInMillis"),
    })

stats_df = pd.DataFrame(query_stats).sort_values("bytes_scanned")
display(stats_df)

partitioned_avg = stats_df[stats_df["query_name"].isin(["q1_high_count", "q2_critical_recent", "q3_join_hot_regions"])]["bytes_scanned"].mean()
full_scan = stats_df.loc[stats_df["query_name"] == "q4_unpartitioned_style_scan", "bytes_scanned"].iloc[0]
reduction_factor = round(full_scan / partitioned_avg, 2) if partitioned_avg else None

print("Full-scan bytes          :", full_scan)
print("Partition-aware avg bytes:", int(partitioned_avg))
print("Approx scan reduction x  :", reduction_factor)


## 7) Athena Workgroup

Create workgroup `citi-analytics` with:
- result location in S3
- **1 GB query data scan limit**
- enforced configuration for predictable team-level cost control


In [ ]:
try:
    athena.create_work_group(
        Name=WORKGROUP,
        Description="Citi analytics team workgroup with query scan limits",
        Configuration={
            "ResultConfiguration": {"OutputLocation": ATHENA_RESULTS_URI},
            "EnforceWorkGroupConfiguration": True,
            "PublishCloudWatchMetricsEnabled": True,
            "BytesScannedCutoffPerQuery": 1_073_741_824,  # 1 GB
        },
        State="ENABLED",
        Tags=[{"Key": "project", "Value": "citi_deep"}],
    )
    print("Created Athena workgroup:", WORKGROUP)
except ClientError as e:
    if e.response["Error"]["Code"] == "InvalidRequestException" and "already exists" in str(e):
        print("Athena workgroup already exists:", WORKGROUP)
    else:
        raise

wg = athena.get_work_group(WorkGroup=WORKGROUP)["WorkGroup"]
print(json.dumps({
    "Name": wg["Name"],
    "State": wg["State"],
    "Configuration": {
        "EnforceWorkGroupConfiguration": wg["Configuration"].get("EnforceWorkGroupConfiguration"),
        "BytesScannedCutoffPerQuery": wg["Configuration"].get("BytesScannedCutoffPerQuery"),
        "ResultConfiguration": wg["Configuration"].get("ResultConfiguration"),
        "PublishCloudWatchMetricsEnabled": wg["Configuration"].get("PublishCloudWatchMetricsEnabled"),
    }
}, indent=2))

wg_sql = f'''
SELECT severity, COUNT(*) AS cnt
FROM {ATHENA_ALERTS_TABLE}
WHERE severity = 'HIGH'
GROUP BY severity
'''.strip()

wg_qid, wg_exec = run_athena(wg_sql, database=ATHENA_DB, workgroup=WORKGROUP)
wg_stats = wg_exec.get("Statistics", {})
print("Workgroup query execution id:", wg_qid)
print("Workgroup data scanned (MB):", bytes_to_mb(wg_stats.get("DataScannedInBytes", 0)))
print("Why this matters: teams get isolated result locations, limits, metrics, and governance.")


## 8) What Just Happened

- **Glue Catalog** became the metadata layer for the lake objects in S3.
- **Glue ETL** job definition was created, with a real script that filters `HIGH/CRITICAL` alerts into Parquet.
- **EMR Serverless** was positioned as Spark without cluster management.
- **Athena** queried the lake directly, and partition-aware access reduced scan size relative to a broad scan.
- **Athena Workgroups** showed how teams can enforce cost controls per workspace.

### Bottom line
Glue Catalog is the metadata layer for AWS analytics. Athena charges per data scanned, so partitioning is often the cleanest 10x–100x cost lever. EMR Serverless gives Spark power without persistent cluster management. Citi-style telemetry and CloudTrail analytics commonly fit this stack well.


## 9) Cleanup

The next cell removes:
- all S3 objects in this run bucket
- Glue table and database
- Glue job
- Athena tables/database
- Athena workgroup

Run it when you are finished.


In [ ]:
def delete_s3_bucket_all_objects(bucket: str):
    paginator = s3.get_paginator("list_objects_v2")
    for page in paginator.paginate(Bucket=bucket):
        contents = page.get("Contents", [])
        if not contents:
            continue
        delete_payload = {"Objects": [{"Key": obj["Key"]} for obj in contents]}
        s3.delete_objects(Bucket=bucket, Delete=delete_payload)
    s3.delete_bucket(Bucket=bucket)

# Drop Athena tables/database first
for stmt in [
    f"DROP TABLE IF EXISTS {ATHENA_DB}.{ATHENA_ALERTS_TABLE}",
    f"DROP TABLE IF EXISTS {ATHENA_DB}.{ATHENA_ENDPOINTS_TABLE}",
]:
    try:
        run_athena(stmt, database=ATHENA_DB, workgroup=WORKGROUP)
    except Exception as e:
        print("Athena cleanup warning:", e)

try:
    run_athena(f"DROP DATABASE IF EXISTS {ATHENA_DB}", workgroup=WORKGROUP)
except Exception as e:
    print("Athena DB cleanup warning:", e)

# Delete Glue table/database
try:
    glue.delete_table(DatabaseName=GLUE_DB, Name=GLUE_ENDPOINTS_TABLE)
    print("Deleted Glue table.")
except ClientError as e:
    print("Glue table cleanup warning:", e)

try:
    glue.delete_database(Name=GLUE_DB)
    print("Deleted Glue database.")
except ClientError as e:
    print("Glue DB cleanup warning:", e)

# Delete Glue job
try:
    glue.delete_job(JobName=GLUE_JOB_NAME)
    print("Deleted Glue job.")
except ClientError as e:
    print("Glue job cleanup warning:", e)

# Delete Athena workgroup
try:
    athena.delete_work_group(
        WorkGroup=WORKGROUP,
        RecursiveDeleteOption=True
    )
    print("Deleted Athena workgroup.")
except ClientError as e:
    print("Athena workgroup cleanup warning:", e)

# Delete bucket last
try:
    delete_s3_bucket_all_objects(S3_BUCKET)
    print("Deleted S3 bucket and all objects.")
except ClientError as e:
    print("S3 cleanup warning:", e)
